# VadCLIP + Temporal Shift-Consistency Trên Colab

Notebook này chỉ đóng vai trò runner để chạy các file Python trong `VadCLIP/src`. Source code
model vẫn nằm trong thư mục `VadCLIP/src`, không được copy trực tiếp vào notebook.

Thứ tự chạy bám theo `instruct.md`:

| Section | Bước trong plan | Nội dung |
|---|---|---|
| 6 | - | Chạy unit test (dataset, loss, batching, smoke) |
| 7 | Bước 1 | Phân bố độ dài video |
| 8 | **Bước 0** | Đo độ nhạy dịch chuyển của baseline — **CỔNG QUYẾT ĐỊNH** |
| 9 | Bước 5 | Kiểm chứng tương đương với `ucf_train.py` |
| 10 | Bước 7 | Baseline đối chứng (`--lambda-consistency 0`) |
| 11 | Bước 6 | Thí nghiệm V0 |
| 12-13 | Bước 6 | Đánh giá + đo lại shift sensitivity |
| 14 | Bước 8 | Nhiều seed |
| 15 | Bước 9 | Các biến thể |
| 16 | Bước 10 | Gom sản phẩm bàn giao |

> **Quan trọng:** Bước 0 (section 8) là cổng quyết định. Theo `instruct.md`: *"Không train gì
> trước khi có kết quả bước này."* Nếu kết quả rơi vào ô `corr > 0.98 và AUC spread < 0.2` thì
> dừng lại và báo cáo, đừng chạy tiếp section 10 trở đi.

## 1. Mount Google Drive Và Cấu Hình Đường Dẫn

Cell này mount Drive, khai báo đường dẫn dùng chung, và định nghĩa các helper
(`run_command`, `build_train_cmd`, `evaluate_model`, ...) cho những cell phía sau.
`run_command` stream output theo từng dòng và lưu log vào `Result/logs_shift_consistency`.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    print('Not running in Colab or Drive is already mounted.')

PROJECT_ROOT = Path('/content/drive/MyDrive/Finetune VadCLIP')
SRC_DIR = PROJECT_ROOT / 'VadCLIP' / 'src'
LIST_DIR = PROJECT_ROOT / 'VadCLIP' / 'list'
DRIVE_FEATURE_ROOT = PROJECT_ROOT / 'UCFClipFeatures'
DRIVE_FEATURE_ARCHIVES = [
    PROJECT_ROOT / 'UCFClipFeatures.tar',
    PROJECT_ROOT / 'UCFClipFeatures.tar.gz',
    PROJECT_ROOT / 'UCFClipFeatures.tgz',
    PROJECT_ROOT / 'UCFClipFeatures.zip',
]
LOCAL_FEATURE_ROOT = Path('/content/UCFClipFeatures')
FEATURE_ROOT = DRIVE_FEATURE_ROOT
PRETRAINED_MODEL = PROJECT_ROOT / 'model_ucf.pth'
RESULT_DIR = PROJECT_ROOT / 'Result'
LOG_DIR = RESULT_DIR / 'logs_shift_consistency'

# Hướng này không dùng description nên train trên list đầy đủ 1610 video.
TRAIN_LIST = '../list/ucf_CLIP_rgb_relative.csv'
TEST_LIST = '../list/ucf_CLIP_rgbtest_relative.csv'
GT_ARGS = [
    '--gt-path', '../list/gt_ucf.npy',
    '--gt-segment-path', '../list/gt_segment_ucf.npy',
    '--gt-label-path', '../list/gt_label_ucf.npy',
]

# instruct.md bước 6 và 7: V0 và baseline đối chứng PHẢI dùng chung giá trị này.
USE_PRETRAINED = False

sys.path.insert(0, str(SRC_DIR))
os.chdir(SRC_DIR)
LOG_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

PY = [sys.executable, '-u']


def run_command(cmd, log_name=None):
    """Chạy lệnh, stream output theo dòng, và lưu log ra Drive."""
    cmd = [str(part) for part in cmd]
    print('$', ' '.join(cmd), flush=True)
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    captured = []
    for line in process.stdout:
        print(line, end='')
        captured.append(line)
    process.wait()
    output = ''.join(captured)
    if log_name:
        (LOG_DIR / log_name).write_text(output, encoding='utf-8')
        print('Log saved:', LOG_DIR / log_name)
    if process.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {process.returncode}')
    return output


def build_train_cmd(tag, lambda_consistency='0.01', seed=234, shift_offset=26, branch='c',
                    random_shift=False, detach=False, warmup=1, max_epoch=10, lr='2e-5',
                    extra=None):
    """Dựng lệnh train. tag quyết định toàn bộ tên file checkpoint/output."""
    return PY + [
        'ucf_train_augment.py',
        '--feature-root', FEATURE_ROOT,
        '--train-list', TRAIN_LIST,
        '--test-list', TEST_LIST,
        *GT_ARGS,
        '--seed', seed,
        '--lambda-consistency', lambda_consistency,
        '--shift-offset', shift_offset,
        '--consistency-branch', branch,
        '--random-shift', str(random_shift).lower(),
        '--consistency-detach', str(detach).lower(),
        '--consistency-warmup', warmup,
        '--max-epoch', max_epoch,
        '--lr', lr,
        '--use-pretrained-model', str(USE_PRETRAINED).lower(),
        '--pretrained-model-path', PRETRAINED_MODEL,
        '--num-workers', 4,
        '--pin-memory', 'true',
        '--eval-steps', 1280,
        '--output-model-path', f'model/model_{tag}.pth',
        '--checkpoint-path', f'model/checkpoint_{tag}.pth',
        '--save-cur-path', f'model/model_cur_{tag}.pth',
        '--epoch-checkpoint-dir', f'model/epoch_checkpoints_{tag}',
    ] + list(extra or [])


def evaluate_model(tag):
    return run_command(PY + [
        'ucf_evaluate.py',
        '--feature-root', FEATURE_ROOT,
        '--test-list', TEST_LIST,
        '--baseline-model-path', PRETRAINED_MODEL,
        '--description-model-path', f'model/model_{tag}.pth',
        '--description-model-type', 'baseline',
        *GT_ARGS,
    ], log_name=f'evaluate_{tag}.log')


def analyze_checkpoints(tag, timeline_count=6):
    output_dir = RESULT_DIR / f'ucf_checkpoint_diagnostics_{tag}'
    return run_command(PY + [
        'ucf_analyze_checkpoints.py',
        '--feature-root', FEATURE_ROOT,
        '--test-list', TEST_LIST,
        '--baseline-model-path', PRETRAINED_MODEL,
        '--epoch-checkpoint-dir', f'model/epoch_checkpoints_{tag}',
        '--description-model-path', f'model/model_{tag}.pth',
        '--finetuned-model-type', 'baseline',
        '--output-dir', output_dir,
        *GT_ARGS,
        '--timeline-count', timeline_count,
    ], log_name=f'analyze_{tag}.log')


def run_shift_sensitivity(model_path, output_name, offsets=(0, 8, 16, 32)):
    output_dir = RESULT_DIR / output_name
    run_command(PY + [
        'ucf_shift_sensitivity.py',
        '--feature-root', FEATURE_ROOT,
        '--test-list', TEST_LIST,
        '--model-path', model_path,
        '--gt-path', '../list/gt_ucf.npy',
        '--offsets', *offsets,
        '--output-dir', output_dir,
    ], log_name=f'{output_name}.log')
    return output_dir


print('Project root :', PROJECT_ROOT)
print('Source dir   :', SRC_DIR)
print('Feature root :', FEATURE_ROOT)
print('Log dir      :', LOG_DIR)
print('Use pretrained checkpoint:', USE_PRETRAINED)

## 2. Cài Đặt Dependencies

Giống notebook description-guided, cộng thêm `pandas` để đọc bảng kết quả.

In [ ]:
!pip -q install ftfy regex tqdm scikit-learn scipy matplotlib pandas

## 3. Kiểm Tra Các File Bắt Buộc

Cell này kiểm tra các file mới của hướng shift-consistency đã được upload lên Drive chưa,
cùng với list, ground truth và checkpoint baseline.

Nếu thiếu ground truth `gt_*.npy`, sinh lại bằng (chạy từ thư mục `VadCLIP`):

```
python list/make_gt_ucf.py        # -> list/gt_ucf.npy
python list/make_gt_mAP_ucf.py    # -> list/gt_label_ucf.npy, list/gt_segment_ucf.npy
```

In [ ]:
required_paths = [
    SRC_DIR / 'model.py',
    SRC_DIR / 'ucf_train.py',
    SRC_DIR / 'ucf_train_augment.py',
    SRC_DIR / 'ucf_option_augment.py',
    SRC_DIR / 'ucf_shift_sensitivity.py',
    SRC_DIR / 'ucf_length_stats.py',
    SRC_DIR / 'ucf_first_step_reference.py',
    SRC_DIR / 'ucf_evaluate.py',
    SRC_DIR / 'ucf_analyze_checkpoints.py',
    SRC_DIR / 'utils' / 'dataset_augment.py',
    SRC_DIR / 'utils' / 'layers.py',
    SRC_DIR / 'tests' / 'test_dataset_augment.py',
    SRC_DIR / 'tests' / 'test_shift_consistency_loss.py',
    SRC_DIR / 'tests' / 'test_two_view_batching.py',
    SRC_DIR / 'tests' / 'test_train_smoke.py',
    LIST_DIR / 'ucf_CLIP_rgb_relative.csv',
    LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv',
    PRETRAINED_MODEL,
]
gt_paths = [
    LIST_DIR / 'gt_ucf.npy',
    LIST_DIR / 'gt_segment_ucf.npy',
    LIST_DIR / 'gt_label_ucf.npy',
]

missing = [str(path) for path in required_paths if not path.exists()]
missing_gt = [str(path) for path in gt_paths if not path.exists()]

if missing:
    print('THIẾU FILE:')
    for path in missing:
        print('  ', path)
if missing_gt:
    print('\nTHIẾU GROUND TRUTH:')
    for path in missing_gt:
        print('  ', path)
    print('  Sinh lại bằng list/make_gt_ucf.py và list/make_gt_mAP_ucf.py (chạy từ thư mục VadCLIP).')

if missing or missing_gt:
    raise FileNotFoundError('Upload các file còn thiếu lên Drive rồi chạy lại cell này.')

print('Đủ toàn bộ file bắt buộc.')
print('Feature root tồn tại:', DRIVE_FEATURE_ROOT.exists())

## 4. Copy Feature Sang Runtime Local

Copy `UCFClipFeatures` từ Drive sang `/content` để giảm I/O. Nên chạy vì hướng này đọc
feature nhiều gấp đôi bình thường (mỗi video sinh hai view, tuy chỉ đọc file một lần).

In [ ]:
import shutil
import time

subprocess.run(['df', '-h', '/content'], check=False)
available_feature_archive = next((path for path in DRIVE_FEATURE_ARCHIVES if path.exists()), None)

start = time.time()
if available_feature_archive is not None:
    local_archive = Path('/content') / available_feature_archive.name
    if not local_archive.exists() or local_archive.stat().st_size != available_feature_archive.stat().st_size:
        print('Copying archive to local runtime:', available_feature_archive)
        shutil.copy2(available_feature_archive, local_archive)
    else:
        print('Local archive already exists with matching size:', local_archive)

    print('Extracting archive:', local_archive)
    if local_archive.suffix == '.zip':
        subprocess.run(['unzip', '-q', '-o', str(local_archive), '-d', '/content'], check=True)
    else:
        subprocess.run(['tar', '-xf', str(local_archive), '-C', '/content'], check=True)
    if not LOCAL_FEATURE_ROOT.exists():
        raise FileNotFoundError('Archive phải chứa thư mục top-level UCFClipFeatures/')
else:
    LOCAL_FEATURE_ROOT.mkdir(parents=True, exist_ok=True)
    if shutil.which('rsync'):
        subprocess.run(['rsync', '-ah', '--info=progress2',
                        f'{DRIVE_FEATURE_ROOT}/', f'{LOCAL_FEATURE_ROOT}/'], check=True)
    else:
        subprocess.run(['cp', '-r', f'{DRIVE_FEATURE_ROOT}/.', str(LOCAL_FEATURE_ROOT)], check=True)

FEATURE_ROOT = LOCAL_FEATURE_ROOT
print(f'\nDone in {time.time() - start:.1f}s. FEATURE_ROOT =', FEATURE_ROOT)

## 5. Kiểm Tra Độ Phủ Feature

Kiểm tra mọi file `.npy` mà list train/test tham chiếu đều tồn tại, tránh lỗi giữa chừng.

In [ ]:
import csv
from collections import Counter


def check_feature_coverage(csv_path, feature_root, preview=30):
    rows = list(csv.DictReader(open(csv_path, encoding='utf-8')))
    missing = [row for row in rows if not (feature_root / row['path']).exists()]
    print(f'{csv_path.name}: rows={len(rows)}, missing_files={len(missing)}')
    if missing:
        print('Missing by label:', dict(Counter(row['label'] for row in missing)))
        for row in missing[:preview]:
            print(f"  {row.get('video_id', '')},{row['label']},{row['path']}")
        raise FileNotFoundError(f'{csv_path.name} tham chiếu file feature không tồn tại.')


for csv_path in [LIST_DIR / 'ucf_CLIP_rgb_relative.csv', LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv']:
    check_feature_coverage(csv_path, FEATURE_ROOT)

## 6. Chạy Unit Test

Bốn bộ test này chạy trong vài giây và không cần feature thật. Chúng xác nhận:

- `test_dataset_augment` — crop đúng vị trí: `feat_shift[j - d] == feat_full[j]`.
- `test_shift_consistency_loss` — `offset=0` cho loss đúng bằng 0, và bản dịch thật sự cũng cho 0
  (nếu căn chỉ số sai thì test này fail).
- `test_two_view_batching` — ghép hai view vào batch không làm đổi output của view đầy đủ.
- `test_train_smoke` — vòng train chạy hết một epoch nhỏ, ghi checkpoint đúng tên.

Nếu có test nào fail thì thường là file trên Drive chưa sync đủ.

In [ ]:
for test_file in [
    'tests/test_dataset_augment.py',
    'tests/test_shift_consistency_loss.py',
    'tests/test_two_view_batching.py',
    'tests/test_train_smoke.py',
]:
    print('=' * 80)
    run_command(PY + [test_file])

## 7. Bước 1 — Phân Bố Độ Dài Video

Số liệu này không ảnh hưởng thiết kế (crop thực hiện *sau* `process_feat`) nhưng cần cho phần
mô tả dữ liệu trong báo cáo. Chú ý dòng `N <= 26`: đó là số video không có vùng chồng lấn ở
offset mặc định, chúng sẽ bị loại khỏi `L_cons`.

In [ ]:
run_command(PY + [
    'ucf_length_stats.py',
    '--feature-root', FEATURE_ROOT,
    '--list-path', TRAIN_LIST,
], log_name='length_stats.log')

## 8. Bước 0 — Đo Độ Nhạy Dịch Chuyển Của Baseline (CỔNG QUYẾT ĐỊNH)

Bước quan trọng nhất, và không train gì cả. Nó chứng minh (hoặc bác bỏ) tiền đề của cả hướng
nghiên cứu: cùng một sự kiện đặt ở offset khác nhau thì VadCLIP có cho điểm khác nhau không.

Script chạy `model_ucf.pth` trên toàn bộ test set với các offset `0 8 16 32`, căn điểm về trục
gốc, rồi tính tương quan Pearson, `mean |Δ|`, `max |Δ|` và AUC theo từng offset.

> Lưu ý khi đọc AUC: theo `instruct.md`, bước này dùng `process_feat` (đường train) chứ không
> phải `process_split` như `ucf_test.py`, nên **AUC ở offset 0 sẽ không đúng bằng 88.02**. Nó chỉ
> là mốc nội bộ. Cổng quyết định dựa trên *spread giữa các offset*, không phải giá trị tuyệt đối.
> Cột `auc_common` tính trên cùng một tập frame cho mọi offset nên so sánh được với nhau;
> `auc_own` tính trên vùng chồng lấn riêng của từng offset.

In [ ]:
BASELINE_SHIFT_DIR = run_shift_sensitivity(
    PRETRAINED_MODEL, 'ucf_shift_sensitivity_baseline', offsets=(0, 8, 16, 32)
)

### 8.1 Đọc Cổng Quyết Định

| Kết quả | Hành động |
|---|---|
| Corr trung bình < 0.95 **hoặc** AUC spread > 0.5% | Tiền đề vững, đi tiếp |
| Corr 0.95–0.98, AUC spread 0.2–0.5% | Có đất nhưng hẹp, đi tiếp với kỳ vọng thấp |
| Corr > 0.98 **và** AUC spread < 0.2% | **Dừng lại và báo cáo trước khi làm gì thêm** |

In [ ]:
import pandas as pd

summary = pd.read_csv(BASELINE_SHIFT_DIR / 'shift_sensitivity_summary.csv')
display(summary)

shifted = summary[summary['offset'] > 0]
mean_corr = float(shifted['mean_classifier_corr'].mean())
has_auc = 'classifier_auc_spread_common' in summary.columns
spread = float(summary['classifier_auc_spread_common'].iloc[0]) if has_auc else float('nan')

print(f'\nCorr trung bình (offset > 0)        : {mean_corr:.4f}')
if has_auc:
    print(f'AUC spread giữa các offset (common) : {spread:.3f} điểm')

if mean_corr < 0.95 or (has_auc and spread > 0.5):
    print('\n=> GO: Tiền đề vững. Chạy tiếp section 9 (bước 5).')
elif mean_corr > 0.98 and has_auc and spread < 0.2:
    print('\n=> STOP: Baseline đã gần như bất biến với dịch chuyển.')
    print('   instruct.md yêu cầu báo cáo lại trước khi làm gì thêm. Đừng chạy section 10+.')
else:
    print('\n=> NARROW: Có đất nhưng hẹp. Đi tiếp nhưng đặt kỳ vọng thấp.')

## 9. Bước 5 — Kiểm Chứng Tính Đúng Đắn Trước Khi Chạy Thật

`ucf_train.py` chỉ log ở step 1280 nên không đọc được loss ở step đầu tiên. `ucf_first_step_reference.py`
tái tạo đúng step đầu tiên của nó (cùng `UCFDataset` gốc, cùng thứ tự khởi tạo, cùng seed, và
import thẳng `CLAS2`/`CLASM` từ `ucf_train`) rồi in ra ba loss.

Cả hai lệnh đều dùng `--num-workers 0` để DataLoader tiêu thụ RNG giống hệt nhau.

**Tiêu chí:** `loss1`, `loss2`, `loss3` trùng nhau tới ít nhất 5 chữ số thập phân. Nếu không
trùng, dừng lại và tìm nguyên nhân trước khi train.

In [ ]:
import re

reference_output = run_command(PY + [
    'ucf_first_step_reference.py',
    '--feature-root', FEATURE_ROOT,
    '--train-list', TRAIN_LIST,
    '--test-list', TEST_LIST,
    '--seed', '234',
], log_name='step5_reference.log')

augment_output = run_command(
    build_train_cmd(tag='step5_check', lambda_consistency='0', seed=234)
    + ['--num-workers', '0', '--eval-steps', '0', '--debug-max-steps', '1'],
    log_name='step5_augment.log',
)


def parse_losses(text, marker):
    pattern = re.escape(marker) + r'\s*loss1=([-\d.eE+]+)\s+loss2=([-\d.eE+]+)\s+loss3=([-\d.eE+]+)'
    match = re.search(pattern, text)
    if not match:
        raise RuntimeError(f'Không tìm thấy dòng "{marker}" trong output.')
    return [float(value) for value in match.groups()]


reference_losses = parse_losses(reference_output, '[reference step 0]')
augment_losses = parse_losses(augment_output, '[step 0]')

print('\n' + '=' * 80)
passed = True
for name, expected, actual in zip(['loss1', 'loss2', 'loss3'], reference_losses, augment_losses):
    difference = abs(expected - actual)
    if difference >= 1e-5:
        passed = False
    print(f'{"OK  " if difference < 1e-5 else "FAIL"} {name}: '
          f'reference={expected:.8f}  augment={actual:.8f}  |diff|={difference:.2e}')

print('\n=> ĐẠT: đường dữ liệu không đổi ngoài ý muốn.' if passed else
      '\n=> KHÔNG ĐẠT: dừng lại, tìm nguyên nhân trước khi train (instruct.md bước 5).')

## 10. Bước 7 — Baseline Đối Chứng (BẮT BUỘC)

Train lại VadCLIP gốc bằng **chính `ucf_train_augment.py` với `--lambda-consistency 0`**, cùng
seed, cùng epoch, cùng lr, cùng `--use-pretrained-model false`.

**Đính chính so với `instruct.md`:** bản plan nói lý do là "số lần đo trên tập test khác nhau".
Điều đó không đúng với setup này. `ucf_train_augment.py` dùng `--eval-steps 1280`, tức đánh giá
mỗi 10 iteration (~12 lần/epoch, ~120 lần cho 10 epoch) và chọn theo classifier AUC — **đúng
cùng cadence và cùng metric** với `ucf_train.py` gốc (`step % 1280`, `AP = AUC`). Luật chọn
model của hai bên là như nhau, nên đó không phải lý do cần baseline đối chứng.

Lý do thật sự là **cô lập biến**. So `v0` với `model_ucf.pth` là so hai thứ khác nhau ở nhiều
chỗ cùng lúc: khởi tạo, seed, môi trường (torch/GPU), và số lần thử mà tác giả đã làm *giữa các
run* trước khi công bố (cái này không quan sát được và không tái lập được). Trong khi
`baseline_ctrl` khác `v0` **đúng một biến duy nhất: `lambda`**.

Bằng chứng vì sao điều này quan trọng: mọi run from-scratch trong `docs/` của dự án đều rơi vào
khoảng 85.8–87.3, không run nào chạm 88.02. Nếu lấy 88.02 làm mốc, phần chênh lệch "do train
from-scratch trong môi trường này" sẽ bị gán nhầm thành "do phương pháp".

Vẫn nên báo cáo 88.02, nhưng ở một dòng riêng ghi rõ "tác giả công bố", không để trong cột so
hơn thua.

In [ ]:
run_command(build_train_cmd(tag='baseline_ctrl', lambda_consistency='0', seed=234),
            log_name='train_baseline_ctrl.log')

## 11. Bước 6 — Thí Nghiệm V0

Cấu hình V0 theo đúng `instruct.md`: `lambda=0.01`, `offset=26` (~10% của 256), offset cố định,
ràng buộc trên nhánh C, không detach, warmup 1 epoch, 10 epoch, lr `2e-5`, train từ scratch.

Áp cho **cả video Normal lẫn Abnormal** — không cần giới hạn ở Normal, vì `L_bce`/`L_nce` chỉ
chạy trên view đầy đủ nên không có rủi ro nhãn nhiễu.

Theo dõi `loss4_raw` trong log:

| Dấu hiệu | Diễn giải |
|---|---|
| `loss4` tụt về ~0 trong vài trăm step, AUC không đổi | Ràng buộc quá dễ thoả. Tăng `shift-offset` hoặc `lambda` |
| `loss4` không giảm | λ quá nhỏ hoặc có bug căn chỉ số. Chạy lại section 6 |
| Shift sensitivity cải thiện rõ, AUC không đổi | **Vẫn là kết quả tốt** và báo cáo được |
| AUC hoặc Ano-AUC tụt > 1% | λ hoặc offset quá lớn. Giảm xuống 0.005 / 16 |

In [ ]:
run_command(build_train_cmd(tag='v0', lambda_consistency='0.01', seed=234,
                            shift_offset=26, branch='c', random_shift=False,
                            detach=False, warmup=1, max_epoch=10, lr='2e-5'),
            log_name='train_v0.log')

## 12. Đánh Giá V0 Và Baseline Đối Chứng

`ucf_evaluate.py` luôn in một dòng `Baseline` từ `model_ucf.pth` — đó là **checkpoint tác giả
công bố**, để tham chiếu chứ không phải cột so hơn thua. Cột so hơn thua là
`baseline_ctrl` (bước 7) so với `v0`.

Cách trình bày trong báo cáo:

```
VadCLIP (tác giả công bố)            88.02      <- dòng tham chiếu, không so hơn thua
VadCLIP from scratch, lambda = 0     xx.xx      <- baseline hợp lệ
+ shift-consistency, lambda = 0.01   yy.yy      <- delta có nghĩa = yy.yy - xx.xx
```

Cả `xx.xx` và `yy.yy` đều là checkpoint tốt nhất trên tập test theo cùng một luật, nên chênh
lệch giữa chúng quy được về đúng một nguyên nhân: `lambda`.

In [ ]:
print('#' * 80)
print('BASELINE ĐỐI CHỨNG (lambda = 0)')
evaluate_model('baseline_ctrl')

print('#' * 80)
print('V0 (lambda = 0.01, offset = 26, branch = c)')
evaluate_model('v0')

In [ ]:
analyze_checkpoints('baseline_ctrl')
analyze_checkpoints('v0')

## 13. Đo Lại Shift Sensitivity Trên Model V0

Theo `instruct.md`, đây là **metric quan trọng nhất của cả hướng này — quan trọng hơn AUC**.
Nếu độ nhạy dịch chuyển giảm rõ mà AUC giữ nguyên, đó vẫn là một kết quả tốt và báo cáo được.

In [ ]:
V0_SHIFT_DIR = run_shift_sensitivity(
    'model/model_v0.pth', 'ucf_shift_sensitivity_v0', offsets=(0, 8, 16, 32)
)
CTRL_SHIFT_DIR = run_shift_sensitivity(
    'model/model_baseline_ctrl.pth', 'ucf_shift_sensitivity_baseline_ctrl', offsets=(0, 8, 16, 32)
)

In [ ]:
import pandas as pd

frames = []
for name, directory in [
    ('paper checkpoint', BASELINE_SHIFT_DIR),
    ('baseline_ctrl', CTRL_SHIFT_DIR),
    ('v0', V0_SHIFT_DIR),
]:
    table = pd.read_csv(Path(directory) / 'shift_sensitivity_summary.csv')
    table.insert(0, 'model', name)
    frames.append(table)

comparison = pd.concat(frames, ignore_index=True)
columns = ['model', 'offset', 'mean_classifier_corr', 'mean_classifier_abs_delta',
           'max_classifier_abs_delta']
if 'classifier_auc_common' in comparison.columns:
    columns += ['classifier_auc_common', 'classifier_auc_spread_common']
display(comparison[columns])

comparison.to_csv(RESULT_DIR / 'shift_sensitivity_comparison.csv', index=False)
print('Saved:', RESULT_DIR / 'shift_sensitivity_comparison.csv')

print('\nCorr trung bình (offset > 0), càng cao càng bất biến:')
for name in ['paper checkpoint', 'baseline_ctrl', 'v0']:
    subset = comparison[(comparison['model'] == name) & (comparison['offset'] > 0)]
    print(f'  {name:18s}: {subset["mean_classifier_corr"].mean():.4f}')

## 14. Bước 8 — Nhiều Seed

Chạy lại **cả baseline (bước 7) lẫn V0 (bước 6)** với ít nhất 3 seed. Báo cáo dạng
`trung bình ± độ lệch chuẩn`.

**Quy tắc đọc:** nếu khoảng cách giữa hai cấu hình nhỏ hơn độ lệch chuẩn thì cải tiến chưa được
chứng minh. Ghi thẳng điều đó vào báo cáo thay vì chọn seed đẹp nhất.

Cell này rất tốn thời gian (6 lần train 10 epoch, chưa kể seed 234 đã chạy ở section 10-11).
Đặt `RUN_MULTI_SEED = True` khi sẵn sàng.

In [ ]:
RUN_MULTI_SEED = False
SEEDS = [234, 1234, 2024]

if not RUN_MULTI_SEED:
    print('Đặt RUN_MULTI_SEED = True để chạy. Seed 234 đã có sẵn từ section 10-11 '
          '(tag baseline_ctrl và v0).')
else:
    for seed in SEEDS:
        for name, lam in [('baseline_ctrl', '0'), ('v0', '0.01')]:
            tag = name if seed == 234 else f'{name}_seed{seed}'
            if Path(f'model/model_{tag}.pth').exists():
                print(f'Bỏ qua {tag}: đã có checkpoint.')
                continue
            print('#' * 80)
            print('Training', tag)
            run_command(build_train_cmd(tag=tag, lambda_consistency=lam, seed=seed),
                        log_name=f'train_{tag}.log')
            evaluate_model(tag)

## 15. Bước 9 — Các Biến Thể

Chạy tuần tự, mỗi lần đổi **một** biến. Điền tên biến thể vào `SELECTED_VARIANTS` rồi chạy.

> **Cảnh báo về V4/V6:** nếu chọn giá trị tốt nhất dựa trên AUC của tập test, bạn đang lặp lại
> đúng lỗi chọn-model-theo-test ở một tầng cao hơn. Muốn chặt chẽ, cắt một validation set từ tập
> train (**chia theo video, không chia theo frame**) và chọn siêu tham số trên đó. Tối thiểu, ghi
> rõ trong báo cáo rằng siêu tham số được chọn trên test.

In [ ]:
VARIANTS = {
    'v1_branch_a':     dict(branch='a'),
    'v2_branch_both':  dict(branch='both'),
    'v3_random_shift': dict(random_shift=True),
    'v4_offset8':      dict(shift_offset=8),
    'v4_offset16':     dict(shift_offset=16),
    'v4_offset51':     dict(shift_offset=51),
    'v5_detach':       dict(detach=True),
    'v6_lambda0.003':  dict(lambda_consistency='0.003'),
    'v6_lambda0.03':   dict(lambda_consistency='0.03'),
    'v6_lambda0.1':    dict(lambda_consistency='0.1'),
}

SELECTED_VARIANTS = []   # ví dụ: ['v1_branch_a', 'v2_branch_both']

if not SELECTED_VARIANTS:
    print('Chưa chọn biến thể nào. Điền tên vào SELECTED_VARIANTS:')
    for name, params in VARIANTS.items():
        print(f'  {name:18s} {params}')
else:
    for name in SELECTED_VARIANTS:
        print('#' * 80)
        print('Variant:', name, VARIANTS[name])
        run_command(build_train_cmd(tag=name, **VARIANTS[name]), log_name=f'train_{name}.log')
        evaluate_model(name)
        analyze_checkpoints(name)
        run_shift_sensitivity(f'model/model_{name}.pth', f'ucf_shift_sensitivity_{name}')

## 16. Bước 10 — Gom Sản Phẩm Bàn Giao

Gom metric của mọi checkpoint đã phân tích thành một bảng so sánh duy nhất, và liệt kê các file
cần cho báo cáo:

1. `shift_sensitivity_summary.csv` trước và sau — bảng chính của hướng này.
2. Bảng metric nhiều seed: baseline vs V0 vs các biến thể.
3. Timeline: đường điểm ở các offset khác nhau, chồng lên ground truth segment.
4. Đường cong AUC/mAP theo epoch (nằm trong thư mục diagnostics).
5. Ghi chú ngắn về những gì **không** hiệu quả — phần này có giá trị không kém phần hiệu quả.

In [ ]:
import pandas as pd

rows = []
for metrics_path in sorted(RESULT_DIR.glob('ucf_checkpoint_diagnostics_*/metrics_by_checkpoint.csv')):
    tag = metrics_path.parent.name.replace('ucf_checkpoint_diagnostics_', '')
    table = pd.read_csv(metrics_path)
    table.insert(0, 'run', tag)
    rows.append(table)

if rows:
    all_metrics = pd.concat(rows, ignore_index=True)
    final_rows = all_metrics[all_metrics['checkpoint'].isin(['baseline', 'final'])]
    display(final_rows)
    all_metrics.to_csv(RESULT_DIR / 'shift_consistency_all_metrics.csv', index=False)
    print('Saved:', RESULT_DIR / 'shift_consistency_all_metrics.csv')
else:
    print('Chưa có thư mục diagnostics nào. Chạy section 12 trước.')

print('\nSản phẩm đã sinh ra:')
for pattern in ['ucf_shift_sensitivity_*/shift_sensitivity_summary.csv',
                'ucf_shift_sensitivity_*/timeline_*.png',
                'ucf_checkpoint_diagnostics_*/metrics_by_checkpoint.csv',
                'logs_shift_consistency/*.log']:
    found = sorted(RESULT_DIR.glob(pattern))
    print(f'  {pattern}: {len(found)} file')